In [118]:
!pip install openpyxl
print("openpyxl installed successfully!")

openpyxl installed successfully!


In [138]:
import pandas as pd

df = pd.read_excel("skull_king_scores.xlsx")

In [139]:
#core scoring
def calculate_score(bid, actual, round_num):
    if bid == 0:
        if actual == 0:
            return 10 * round_num
        else:
            return -10 * round_num
    elif bid == actual:
        return 20 * bid
    else:
        return -10 * abs(bid - actual)

In [140]:
#each round
def play_round(round_num):
    players = [df.iloc[0,1], df.iloc[1,1], df.iloc[2,1], df.iloc[3,1], df.iloc[4,1], df.iloc[5,1]]
    bids = [df.iloc[6*round_num - 6, 2], df.iloc[6*round_num - 5, 2], df.iloc[6*round_num - 4, 2], df.iloc[6*round_num - 3, 2], df.iloc[6*round_num - 2, 2], df.iloc[6*round_num - 1, 2]]
    tricks = [df.iloc[6*round_num - 6, 3], df.iloc[6*round_num - 5, 3], df.iloc[6*round_num - 4, 3], df.iloc[6*round_num - 3, 3], df.iloc[6*round_num - 2, 3], df.iloc[6*round_num - 1, 3]]
    bonuses = [df.iloc[6*round_num - 6, 4], df.iloc[6*round_num - 5, 4], df.iloc[6*round_num - 4, 4], df.iloc[6*round_num - 3, 4], df.iloc[6*round_num - 2, 4], df.iloc[6*round_num - 1, 4]]
    scores = []
    for i in range(6):
        player = players[i]
        bid = bids[i]
        actual = tricks[i]
        bonus = bonuses[i]
        current_score = calculate_score(bid, actual, round_num) + bonus
        scores.append(current_score)
    return scores

In [141]:
# game loop
def play_game():
    for i in range(10):
        scores = play_round(i+1)
        round_num = i+1
        for j in range(6):
            df.at[6*round_num - 6+j, "SCORE"] = scores[j]
            if round_num >1:
                df.at[6*round_num - 6+j, "TOTAL"] = df.at[6*(round_num-1) - 6+j, "TOTAL"] + scores[j]
            else:
                df.at[6*round_num - 6+j, "TOTAL"] = scores[j]
    return 'scores updated'
play_game()

'scores updated'

In [142]:
#results
df.to_excel("skull_king_scores.xlsx", index=False)

In [149]:
def generate_summary(df):
    # average score per player
    avg_scores = df.groupby("PLAYER")["SCORE"].mean().rename("AVG_SCORE")

    # correct bid percentage
    df["CORRECT_BID"] = df["BID"] == df["TRICKS"]
    correct_bid_pct = df.groupby("PLAYER")["CORRECT_BID"].mean().rename("CORRECT_BID_PCT")

    # total score
    final_scores = df.groupby("PLAYER")["SCORE"].sum().rename("TOTAL_SCORE")
    
    # top scorer
    top_scorer = final_scores.idxmax()
    top_score = final_scores.max()
    print(f"🏆 Top Scorer: {top_scorer} with {top_score} points")

    # win rate: how many rounds each player won 
    round_max = df.groupby("ROUND")["SCORE"].transform("max")
    df["WON_ROUND"] = df["SCORE"] == round_max
    win_rates = df.groupby("PLAYER")["WON_ROUND"].mean().rename("WIN_RATE")

    # average bid deviation
    df["BID_DEV"] = (df["BID"] - df["TRICKS"]).abs()
    avg_dev = df.groupby("PLAYER")["BID_DEV"].mean().rename("AVG_BID_DEV")

    # summary table
    summary = pd.concat([
        avg_scores,
        correct_bid_pct,
        win_rates,
        avg_dev,
        final_scores
    ], axis=1)

    # rank
    summary["RANK"] = summary["TOTAL_SCORE"].rank(ascending=False).astype(int)
    summary = summary.sort_values("RANK")

    return summary

In [158]:
#graph
import matplotlib.pyplot as plt

## cumulative total per player
df_sorted = df.sort_values(by=["PLAYER", "ROUND"])
df_sorted["CUM_TOTAL"] = df_sorted.groupby("PLAYER")["SCORE"].cumsum()

pivot = df_sorted.pivot(index="ROUND", columns="PLAYER", values="CUM_TOTAL")

plt.figure(figsize=(10, 6))

for player in pivot.columns:
    plt.plot(pivot.index, pivot[player], marker='o', label=player)

plt.title("Skull King: Cumulative Scores per Round")
plt.xlabel("Round")
plt.ylabel("Total Score")
plt.grid(True)
plt.legend(title="Player")
plt.tight_layout()

plt.savefig("score_chart.png")
plt.close()

from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# game statistics
summary_df = generate_summary(df)
print(summary_df)

with pd.ExcelWriter("skull_king_scores_combined.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Scores", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=True)
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# workbook
wb = load_workbook("skull_king_scores_combined.xlsx")

graph_sheet = wb.create_sheet("Score Graph")

img = Image("score_chart.png")
img.width = 640
img.height = 480

graph_sheet.add_image(img, "B2")

wb.save("skull_king_scores_combined.xlsx")

🏆 Top Scorer: F with 300 points
        AVG_SCORE  CORRECT_BID_PCT  WIN_RATE  AVG_BID_DEV  TOTAL_SCORE  RANK
PLAYER                                                                      
F            30.0              0.6       0.4          0.5          300     1
C            23.0              0.6       0.3          0.5          230     2
E            13.0              0.6       0.0          0.5          130     3
B             3.0              0.3       0.1          0.9           30     4
D            -7.0              0.6       0.3          0.4          -70     5
A            -8.0              0.3       0.3          0.9          -80     6
